# AgenPingui - Enfoque con agentes

Este notebook implementa una version guiada del enfoque con agentes para el dataset de pinguinos.

Objetivos:
- Crear una carpeta independiente para el enfoque con agentes.
- Generar un archivo de guia del agente.
- Construir un runner en Python que calcule, genere graficos y escriba artifacts.
- Ejecutar las fases OBSERVE, DESCRIBE y opcionalmente TEST.
- Preparar prompts para la fase HYPOTHESIZE_AND_CONCLUDE.

## Arquitectura

Antes de usar el agente, se debe tener claro que:

- Runner: script en Python que calcula, genera graficos y escribe artifacts.
- Artifacts: archivos JSON o PNG con resultados verificables. Son la unica fuente valida de evidencia.
- Agente: propone acciones y redacta hipotesis o conclusiones con base en artifacts.

Fases del flujo:
- OBSERVE
- DESCRIBE
- HYPOTHESIZE_AND_CONCLUDE

In [8]:
from pathlib import Path
import json
import textwrap

base_dir = Path.cwd() / 'agente_pingui'
prompts_dir = base_dir / 'Promts'
artifacts_dir = base_dir / 'artifacts'
base_dir.mkdir(exist_ok=True)
prompts_dir.mkdir(exist_ok=True)
artifacts_dir.mkdir(exist_ok=True)

print('Carpeta base:', base_dir)
print('Subcarpetas creadas:', prompts_dir, artifacts_dir)

Carpeta base: c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui
Subcarpetas creadas: c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui\Promts c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui\artifacts


## Guia del agente

En cada interaccion debe definirse con claridad:
- La fase actual: OBSERVE, DESCRIBE o HYPOTHESIZE_AND_CONCLUDE.
- Que artifacts estan disponibles.
- Que afirmaciones deben sustentarse solo con artifacts.

In [9]:
guide_text = textwrap.dedent('''
FASE: {FASE_ACTUAL}

Artifacts disponibles:
{ARTIFACTS_DISPONIBLES}

Reglas:
1. No inventar valores.
2. Usar solo evidencia contenida en artifacts.
3. Proponer acciones concretas para el runner.
4. En DESCRIBE, seleccionar graficos de forma exploratoria, sin sesgo predefinido.
5. En HYPOTHESIZE_AND_CONCLUDE, formular hipotesis falsables y conclusiones sin afirmar causalidad.

Salida esperada del agente:
- Plan de acciones para el runner.
- Lista de artifacts que deberian generarse.
- Observaciones, hipotesis o conclusiones segun la fase.
''').strip()

guide_path = prompts_dir / 'hola_agentes.txt'
guide_path.write_text(guide_text, encoding='utf-8')
print('Guia creada en:', guide_path)
print(guide_path.read_text(encoding='utf-8'))

Guia creada en: c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui\Promts\hola_agentes.txt
FASE: {FASE_ACTUAL}

Artifacts disponibles:
{ARTIFACTS_DISPONIBLES}

Reglas:
1. No inventar valores.
2. Usar solo evidencia contenida en artifacts.
3. Proponer acciones concretas para el runner.
4. En DESCRIBE, seleccionar graficos de forma exploratoria, sin sesgo predefinido.
5. En HYPOTHESIZE_AND_CONCLUDE, formular hipotesis falsables y conclusiones sin afirmar causalidad.

Salida esperada del agente:
- Plan de acciones para el runner.
- Lista de artifacts que deberian generarse.
- Observaciones, hipotesis o conclusiones segun la fase.


## Construccion del runner

El runner debe poder:
- Cargar el dataset.
- Ejecutar funciones basicas de observacion.
- Ampliarse luego con descripcion, visualizacion y pruebas.
- Escribir artifacts en formato JSON y PNG.

In [10]:
runner_code = textwrap.dedent('''
from __future__ import annotations

import itertools
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

BASE_DIR = Path(__file__).resolve().parent
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

def load_dataset() -> pd.DataFrame:
    return sns.load_dataset('penguins')

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

def observe_phase(df: pd.DataFrame) -> dict:
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]
    payload = {
        'shape': {'rows': int(df.shape[0]), 'columns': int(df.shape[1])},
        'numeric_columns': num_cols,
        'categorical_columns': cat_cols,
        'missing_by_column': df.isna().sum().to_dict(),
        'duplicate_rows': int(df.duplicated().sum()),
        'low_cardinality': {c: int(df[c].nunique(dropna=True)) for c in cat_cols if df[c].nunique(dropna=True) <= 10}
    }
    write_json(ARTIFACTS_DIR / '00_raw_profile.json', payload)
    return payload

def describe_phase(df: pd.DataFrame) -> dict:
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]

    numeric_summary = {}
    for col in num_cols:
        s = df[col].dropna()
        numeric_summary[col] = {
            'mean': float(s.mean()),
            'median': float(s.median()),
            'std': float(s.std()),
            'iqr': float(s.quantile(0.75) - s.quantile(0.25))
        }

    categorical_summary = {}
    for col in cat_cols:
        counts = df[col].astype('object').where(df[col].notna(), 'Missing').value_counts(dropna=False)
        total = int(counts.sum())
        categorical_summary[col] = {
            key: {
                'count': int(value),
                'percentage': round((int(value) / total) * 100, 2)
            }
            for key, value in counts.items()
        }

    crosstabs = {}
    for c1, c2 in itertools.combinations(cat_cols, 2):
        crosstabs[f'{c1}__vs__{c2}'] = pd.crosstab(df[c1], df[c2], dropna=False).to_dict()

    correlations = {
        'pearson': df[num_cols].corr(method='pearson').round(4).to_dict(),
        'spearman': df[num_cols].corr(method='spearman').round(4).to_dict()
    }

    descriptive_payload = {
        'numeric_summary': numeric_summary,
        'categorical_summary': categorical_summary,
        'crosstabs': crosstabs,
        'correlations': correlations
    }
    write_json(ARTIFACTS_DIR / '04_descriptive_stats.json', descriptive_payload)

    visual_registry = []
    low_card_cols = [c for c in cat_cols if df[c].nunique(dropna=True) <= 10]
    for col in low_card_cols:
        series = df[col].astype('object').where(df[col].notna(), 'Missing')
        order = series.value_counts().index
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.countplot(x=series, order=order, ax=ax)
        ax.set_title(f'Conteo de {col}')
        ax.tick_params(axis='x', rotation=30)
        fig.tight_layout()
        out = ARTIFACTS_DIR / f'count_{col}.png'
        fig.savefig(out, dpi=150)
        plt.close(fig)
        visual_registry.append({'type': 'countplot', 'column': col, 'file': out.name})

    for col in num_cols:
        fig, ax = plt.subplots(figsize=(7, 4))
        sns.histplot(df[col].dropna(), kde=True, bins=20, ax=ax)
        ax.set_title(f'Histograma de {col}')
        fig.tight_layout()
        out = ARTIFACTS_DIR / f'hist_{col}.png'
        fig.savefig(out, dpi=150)
        plt.close(fig)
        visual_registry.append({'type': 'histogram', 'column': col, 'file': out.name})

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(data=df, x='species', y='bill_length_mm', ax=ax)
    ax.set_title('bill_length_mm por species')
    fig.tight_layout()
    out = ARTIFACTS_DIR / 'box_bill_length_species.png'
    fig.savefig(out, dpi=150)
    plt.close(fig)
    visual_registry.append({'type': 'boxplot', 'columns': ['species', 'bill_length_mm'], 'file': out.name})

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=df, x='flipper_length_mm', y='body_mass_g', hue='species', alpha=0.85, ax=ax)
    ax.set_title('flipper_length_mm vs body_mass_g por species')
    fig.tight_layout()
    out = ARTIFACTS_DIR / 'scatter_flipper_bodymass_species.png'
    fig.savefig(out, dpi=150)
    plt.close(fig)
    visual_registry.append({'type': 'scatter', 'columns': ['flipper_length_mm', 'body_mass_g', 'species'], 'file': out.name})

    fig, ax = plt.subplots(figsize=(7, 5))
    corr = df[num_cols].corr(method='pearson')
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, ax=ax)
    ax.set_title('Heatmap de correlacion (Pearson)')
    fig.tight_layout()
    out = ARTIFACTS_DIR / 'heatmap_pearson.png'
    fig.savefig(out, dpi=150)
    plt.close(fig)
    visual_registry.append({'type': 'heatmap', 'method': 'pearson', 'file': out.name})

    write_json(ARTIFACTS_DIR / '05_visual_registry.json', {'visuals': visual_registry})
    return descriptive_payload

def test_phase(df: pd.DataFrame) -> dict:
    alpha = 0.05
    tmp = df[['flipper_length_mm', 'body_mass_g']].dropna()
    rho, p1 = stats.spearmanr(tmp['flipper_length_mm'], tmp['body_mass_g'])

    tmp2 = df[['bill_length_mm', 'species']].dropna()
    groups = [g['bill_length_mm'].values for _, g in tmp2.groupby('species')]
    H, p2 = stats.kruskal(*groups)

    tmp3 = df[['species', 'island']].dropna()
    table = pd.crosstab(tmp3['species'], tmp3['island'])
    chi2, p3, dof, _ = stats.chi2_contingency(table)

    payload = {
        'alpha': alpha,
        'tests': [
            {
                'hypothesis': 'flipper_length_mm asociado con body_mass_g',
                'test': 'Spearman',
                'statistic': float(rho),
                'p_value': float(p1),
                'supports_hypothesis': bool(p1 < alpha)
            },
            {
                'hypothesis': 'bill_length_mm difiere por species',
                'test': 'Kruskal-Wallis',
                'statistic': float(H),
                'p_value': float(p2),
                'supports_hypothesis': bool(p2 < alpha)
            },
            {
                'hypothesis': 'species asociado con island',
                'test': 'Chi-cuadrado',
                'statistic': float(chi2),
                'degrees_of_freedom': int(dof),
                'p_value': float(p3),
                'supports_hypothesis': bool(p3 < alpha)
            }
        ]
    }
    write_json(ARTIFACTS_DIR / '08_tests.json', payload)
    return payload

if __name__ == '__main__':
    df = load_dataset()
    observe_phase(df)
    describe_phase(df)
    test_phase(df)
''').strip()

runner_path = base_dir / 'runner.py'
runner_path.write_text(runner_code, encoding='utf-8')
print('Runner creado en:', runner_path)

Runner creado en: c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui\runner.py


## Ejecucion de la fase OBSERVE

El proceso comienza preparando el runner y generando el primer artifact:
- `artifacts/00_raw_profile.json`

En esta fase se le indica al agente que no hay artifacts descriptivos disponibles todavia.

In [11]:
import subprocess
import sys

observe_code = textwrap.dedent('''
from runner import load_dataset, observe_phase
df = load_dataset()
payload = observe_phase(df)
print(payload)
''').strip()

result = subprocess.run(
    [sys.executable, '-c', observe_code],
    cwd=base_dir,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

{'shape': {'rows': 344, 'columns': 7}, 'numeric_columns': ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], 'categorical_columns': ['species', 'island', 'sex'], 'missing_by_column': {'species': 0, 'island': 0, 'bill_length_mm': 2, 'bill_depth_mm': 2, 'flipper_length_mm': 2, 'body_mass_g': 2, 'sex': 11}, 'duplicate_rows': 0, 'low_cardinality': {'species': 3, 'island': 3, 'sex': 2}}



## Prompt sugerido para OBSERVE

Fase actual: OBSERVE

Artifacts disponibles:
- Ninguno, excepto el contexto del dataset y la posibilidad de generar `00_raw_profile.json`.

Solicitud al agente:
- Proponer un plan de observacion inicial.
- Indicar que informacion debe producir el runner.
- No formular conclusiones causales.

## Ejecucion de la fase DESCRIBE

Luego se amplia el runner con funciones descriptivas para generar:
- `artifacts/04_descriptive_stats.json`
- `artifacts/05_visual_registry.json`

En esta fase el agente ya dispone del perfil del dataset.

In [12]:
describe_code = textwrap.dedent('''
from runner import load_dataset, describe_phase
df = load_dataset()
payload = describe_phase(df)
print(payload.keys())
''').strip()

result = subprocess.run(
    [sys.executable, '-c', describe_code],
    cwd=base_dir,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

dict_keys(['numeric_summary', 'categorical_summary', 'crosstabs', 'correlations'])



## Prompt sugerido para DESCRIBE

Fase actual: DESCRIBE

Artifacts disponibles:
- `00_raw_profile.json`

Solicitud al agente:
- Proponer analisis descriptivos adicionales.
- Seleccionar graficos exploratorios sin sesgo predefinido.
- Indicar que nuevos artifacts debe producir el runner.

## Fase HYPOTHESIZE_AND_CONCLUDE

Con los artifacts descriptivos disponibles, el agente debe formular:
- Hipotesis falsables.
- Conclusiones descriptivas.
- Preguntas para un humano.

Los resultados deben guardarse idealmente en:
- `artifacts/06_hypotheses_log.json`
- `artifacts/07_conclusions.json`
- `artifacts/09_questions.json`

Todas las afirmaciones deben estar sustentadas en artifacts.

In [13]:
hypotheses_payload = {
    'phase': 'HYPOTHESIZE_AND_CONCLUDE',
    'available_artifacts': [
        '00_raw_profile.json',
        '04_descriptive_stats.json',
        '05_visual_registry.json'
    ],
    'hypotheses_examples': [
        'flipper_length_mm se asocia con body_mass_g',
        'bill_length_mm difiere por species',
        'species se asocia con island'
    ],
    'instructions': [
        'No afirmar causalidad',
        'Sustentar cada afirmacion con artifacts',
        'Proponer pruebas estadisticas si hacen falta'
    ]
}

out_path = artifacts_dir / '06_hypotheses_log.json'
out_path.write_text(json.dumps(hypotheses_payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('Seed de hipotesis guardado en:', out_path)
print(json.dumps(hypotheses_payload, indent=2, ensure_ascii=False))

Seed de hipotesis guardado en: c:\Users\Asus\Downloads\Juan\compu2\Laboratorios\agente_pingui\artifacts\06_hypotheses_log.json
{
  "phase": "HYPOTHESIZE_AND_CONCLUDE",
  "available_artifacts": [
    "00_raw_profile.json",
    "04_descriptive_stats.json",
    "05_visual_registry.json"
  ],
  "hypotheses_examples": [
    "flipper_length_mm se asocia con body_mass_g",
    "bill_length_mm difiere por species",
    "species se asocia con island"
  ],
  "instructions": [
    "No afirmar causalidad",
    "Sustentar cada afirmacion con artifacts",
    "Proponer pruebas estadisticas si hacen falta"
  ]
}


## Pruebas estadisticas opcionales

Opcionalmente pueden ejecutarse pruebas y registrar los resultados en `artifacts/08_tests.json`.

In [14]:
test_code = textwrap.dedent('''
from runner import load_dataset, test_phase
df = load_dataset()
payload = test_phase(df)
print(payload)
''').strip()

result = subprocess.run(
    [sys.executable, '-c', test_code],
    cwd=base_dir,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

{'alpha': 0.05, 'tests': [{'hypothesis': 'flipper_length_mm asociado con body_mass_g', 'test': 'Spearman', 'statistic': 0.8399741230312999, 'p_value': 2.7632189971796422e-92, 'supports_hypothesis': True}, {'hypothesis': 'bill_length_mm difiere por species', 'test': 'Kruskal-Wallis', 'statistic': 244.13671803364164, 'p_value': 9.691371997194331e-54, 'supports_hypothesis': True}, {'hypothesis': 'species asociado con island', 'test': 'Chi-cuadrado', 'statistic': 299.55032743148195, 'degrees_of_freedom': 4, 'p_value': 1.354573829719252e-63, 'supports_hypothesis': True}]}



## Interaccion con el agente

El flujo es ciclico:
1. Solicitar al agente un plan acorde con la fase actual.
2. Ejecutar las acciones con el runner.
3. Guardar artifacts.
4. Repetir el ciclo en la siguiente fase.

En DESCRIBE, los graficos deben seleccionarse de forma exploratoria, sin sesgo predefinido.
En HYPOTHESIZE_AND_CONCLUDE, todas las afirmaciones deben estar sustentadas en artifacts.

## Generalizacion

Una vez completado el flujo con el dataset de pinguinos, se debe repetir el proceso con otro dataset.

Objetivo:
- Verificar que la arquitectura funciona de forma general.
- Evaluar que partes del prompt deben ajustarse para otros contextos.

## Entregables

- [x] Reporte final en HTML: `agente_pingui/artifacts/reporte_final.html`
- [x] Resumen en Markdown: `agente_pingui/artifacts/resumen.md`
- [x] Hipotesis y conclusiones documentadas: `agente_pingui/artifacts/07_conclusions.json`
- [x] Registro de interaccion con el agente (prompts y decisiones): `agente_pingui/artifacts/registro_interaccion_agente.md`

Sugerencia: el reporte final se compilo usando evidencia de los artifacts del runner (00, 04, 05, 06 y 08).